# State concept in Langchain

## Full example
State is the shared memory that flows through your agent or workflow. It's a single, centralized data structure that every step in your workflow can read from and write to.

<img src="./images/states.png" width="2000" height="100">

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
from langchain.chat_models import init_chat_model

_ = load_dotenv(find_dotenv())

api_key = os.getenv("OPENAI_API_KEY")
base_url = os.getenv("BASE_URL")

llm = init_chat_model("gpt-4o-mini", model_provider="openai", temperature=0, api_key=api_key, base_url=base_url)
ollama = init_chat_model("llama3.1:8b ", model_provider="ollama", temperature=0)


from typing import TypedDict, Annotated, List
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, ToolMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain.agents import create_agent

# ============================================================
# 1. Define State
# ============================================================

class ChatState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]
    user_name: str
    conversation_topic: str
    tool_executions: List[dict]
    
# ============================================================
# 2. Define Tools
# ============================================================

@tool
def get_user_info(user_id: str) -> str:
    """Get information about a user"""
    # Simulated user data
    users = {
        "user_123": {"name": "Alice", "preferences": ["tech", "AI"]},
        "user_456": {"name": "Bob", "preferences": ["sports", "movies"]}
    }
    return f"User info: {users.get(user_id, {}).get('name', 'Unknown')}"

@tool
def search_knowledge_base(query: str) -> str:
    """Search the knowledge base"""
    # Simulated search
    return f"Search results for '{query}': Found 3 relevant documents"

# ============================================================
# 3. Create Agent with State
# ============================================================

model = ChatOpenAI(model="gpt-4o")
tools = [get_user_info, search_knowledge_base]

agent = create_agent(
    model=ollama,
    tools=tools,
    system_prompt="You are a helpful assistant with access to user info and knowledge base."
)

# ============================================================
# 4. Run with State Tracking
# ============================================================

async def run_chat():
    # Initial state
    state = ChatState(
        messages=[HumanMessage(content="Hi, I'm Alice!")],
        user_name="Alice",
        conversation_topic="",
        tool_executions=[]
    )
    
    print("Chat started!")
    print("="*40)
    
    # Simulate conversation
    queries = [
        "What can you tell me about AI?",
        "Can you check my user info?",
        "Thanks! That was helpful."
    ]
    
    for query in queries:
        print(f"\n👤 User: {query}")
        
        # Update state with user message
        state["messages"].append(HumanMessage(content=query))
        
        # Get response from agent
        result = await agent.ainvoke({
            "messages": state["messages"]
        })
        
        # Update state with response
        state["messages"] = result["messages"]
        
        # Track tool executions
        for msg in result["messages"]:
            if hasattr(msg, "tool_calls") and msg.tool_calls:
                state["tool_executions"].extend(msg.tool_calls)
        
        # Show response
        print(f"🤖 Assistant: {result['messages'][-1].content}")
        
        # Update topic
        if "AI" in query:
            state["conversation_topic"] = "Artificial Intelligence"
    
    # Final state summary
    print("\n" + "="*40)
    print("📊 CONVERSATION SUMMARY")
    print("="*40)
    print(f"User: {state['user_name']}")
    print(f"Topic: {state['conversation_topic']}")
    print(f"Messages: {len(state['messages'])}")
    print(f"Tool executions: {len(state['tool_executions'])}")

# Run the chat
import asyncio
asyncio.run(run_chat())

## Start state

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.agents import AgentState

class CustomState(AgentState):
    favourite_colour: str

## Write to state

In [3]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def update_favourite_colour(favourite_colour: str, runtime: ToolRuntime) -> Command:
    """Update the favourite colour of the user in the state once they've revealed it."""
    return Command(update={
        "favourite_colour": favourite_colour, 
        "messages": [ToolMessage("Successfully updated favourite colour", tool_call_id=runtime.tool_call_id)]}
        )

In [9]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    #model = "gpt-5-nano",
    model = ollama,
    tools=[update_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [10]:
from langchain.messages import HumanMessage

response = agent.invoke(
    { "messages": [HumanMessage(content="My favourite colot is green")]},
    {"configurable": {"thread_id": "1"}}
)

In [11]:
print(response["messages"][-1].content)

Your favourite colour has been updated to green.


In [12]:
response = agent.invoke(
    { 
        "messages": [HumanMessage(content="Hi, how are you?")],
        "favourite_colour": "red"
    },
    {"configurable": {"thread_id": "10"}}
)

print(response["messages"][-1].content)

I'm not able to provide a function call response for the given prompt as it doesn't match any of the provided functions. The prompt is a greeting, and there is no function that matches this type of input.


In [14]:
response = agent.invoke(
    { "messages": [HumanMessage(content="What is my favourite color?")]},
    {"configurable": {"thread_id": "1"}}
)
print(response["messages"][-1].content)

{"name": "get_favourite_colour", "parameters": {}}


## Read state

In [15]:
@tool
def read_favourite_colour(runtime: ToolRuntime) -> str:
    """Read the favourite colour of the user from the state."""
    try:
        return runtime.state["favourite_colour"]
    except KeyError:
        return "No favourite colour found in state"

agent = create_agent(
    model = ollama,
    tools=[update_favourite_colour, read_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [18]:
response = agent.invoke(
    { "messages": [HumanMessage(content="My favorite color is green")]},
    {"configurable": {"thread_id": "1"}}
)

print(response["messages"][-1].content)

Your favorite color is green.


In [19]:
response = agent.invoke(
    { "messages": [HumanMessage(content="What is my favorite color?")]},
    {"configurable": {"thread_id": "1"}}
)

print(response["messages"][-1].content)

Your favorite color is green.


In [21]:
response

{'messages': [HumanMessage(content='رنگ مورد علاقه من سبز است', additional_kwargs={}, response_metadata={}, id='fe0da5a8-4307-4872-81c7-e2f023191bd6'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 284, 'prompt_tokens': 165, 'total_tokens': 449, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': None, 'reasoning_tokens': 256, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-Dqa7rJA8OaRoNeEwwQToICQJBO3uB', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019ec529-8c1e-73a0-9b29-8fa0be01f1c4-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'favourite_colour': 'سبز'}, 'id': 'call_XkxcDynSdg2tYmUjDBJcJ6os', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 165, 'output_token